# Tests

This notebook consolidates the PENRS test suite from eight source files: `test_arbiter`, `test_cache`, `test_pipeline`, `test_rate_limit`, `test_router`, `test_step1`, `test_step2`, and `test_worker`. Tests are organized into labeled sections by source module, and the final cell invokes `pytest` to run the full suite and print a pass/fail summary.

In [ ]:
# Standard library
import asyncio
import importlib
import json
import os
import uuid
from datetime import datetime, timedelta, timezone
from enum import Enum
from pathlib import Path
from unittest.mock import patch

# Third-party
import httpx
import pytest

# Project — utility module (consolidates penrs_cache, penrs_http, penrs_rate_limit, penrs_router)
import utils

# Aliases so existing test helpers retain their patching targets:
#   _reload_cache_module reloads utils and accesses utils.cache_*, utils.logger, etc.
#   Step 2 tests patch utils.httpx.AsyncClient, utils.asyncio.sleep, utils.logger, etc.
penrs_cache = utils
penrs_http = utils

# Notebook-resident symbols (migrated from deleted .py source files):
#   PENRSWorker, truncate_for_context  ← worker_nodes.ipynb
#   ArbiterAgent, ARBITER_SYSTEM_PROMPT, run_all_workers, run_penrs  ← orchestrator.ipynb
%run worker_nodes.ipynb
%run orchestrator.ipynb

# Project — symbols from utils
from utils import DOCUMENT_API_ROUTING, DocumentType, penrs_fetch_document

# Test support — MCP server import helper used by fetcher tests
from tests.test_support import import_penrs_server, import_worker_nodes

## Arbiter Tests

In [2]:
def _worker_result(
    *,
    name: str,
    weight: float,
    signal_density: float,
    score: float,
    star_rating: int | None = None,
    summary: str = "",
) -> dict:
    result_payload = {"score": score, "summary": summary}
    if star_rating is not None:
        result_payload["star_rating"] = star_rating
    return {
        "status": "available",
        "worker": {
            "name": name,
            "weight": weight,
            "signal_density": signal_density,
        },
        "result": result_payload,
    }


def test_system_prompt_contains_required_role_and_mandatory_contradictions():
    assert "Lead Portfolio Manager" in ARBITER_SYSTEM_PROMPT
    assert "Lipstick on a Pig" in ARBITER_SYSTEM_PROMPT
    assert "Bailing Out" in ARBITER_SYSTEM_PROMPT
    assert "Dilute and Delay" in ARBITER_SYSTEM_PROMPT


def test_arbiter_validates_required_worker_schema_fields():
    arbiter = ArbiterAgent()
    bad_worker = {
        "status": "available",
        "worker": {
            "name": "MissingWeight",
            "signal_density": 0.8,
        },
        "result": {"score": 0.3},
    }

    with pytest.raises(ValueError, match="worker.weight"):
        arbiter.evaluate([bad_worker])


def test_arbiter_clamps_scores_and_applies_star_rating_weights():
    arbiter = ArbiterAgent()
    workers = [
        _worker_result(
            name="WorkerA",
            weight=2.0,
            signal_density=0.9,
            score=1.8,
            star_rating=5,
        ),
        _worker_result(
            name="WorkerB",
            weight=1.5,
            signal_density=0.2,
            score=-2.4,
            star_rating=2,
        ),
    ]

    report = arbiter.evaluate(workers)
    worker_scores = {entry["name"]: entry for entry in report["worker_scores"]}

    assert worker_scores["WorkerA"]["normalized_score"] == 1.0
    assert worker_scores["WorkerA"]["effective_weight"] == 2.0
    assert worker_scores["WorkerA"]["weighted_score"] == 2.0

    assert worker_scores["WorkerB"]["normalized_score"] == -1.0
    assert worker_scores["WorkerB"]["effective_weight"] == 0.6
    assert worker_scores["WorkerB"]["weighted_score"] == -0.6

    assert report["weighted_score"] == pytest.approx(1.4 / 2.6, rel=1e-6)


def test_arbiter_returns_json_schema_with_contradiction_flags_and_severities():
    arbiter = ArbiterAgent()
    workers = [
        _worker_result(
            name="NarrativeWorker",
            weight=1.0,
            signal_density=0.6,
            score=0.2,
            summary="Management is putting lipstick on a pig while bailing out early holders.",
        )
    ]

    report = arbiter.evaluate(workers)
    contradictions = {entry["name"]: entry for entry in report["contradictions"]}

    assert set(contradictions) == {"Lipstick on a Pig", "Bailing Out", "Dilute and Delay"}
    assert contradictions["Lipstick on a Pig"]["flagged"] is True
    assert contradictions["Lipstick on a Pig"]["severity"] == "High"
    assert contradictions["Bailing Out"]["flagged"] is True
    assert contradictions["Bailing Out"]["severity"] == "High"
    assert contradictions["Dilute and Delay"]["severity"] == "Medium"

    json.dumps(report)

## Cache Tests

In [3]:
TEST_FILES_DIR = (Path.cwd() / "Test_files").resolve()


def local_tmp_dir() -> Path:
    TEST_FILES_DIR.mkdir(parents=True, exist_ok=True)
    tmp_dir = TEST_FILES_DIR / f"test_tmp_{uuid.uuid4().hex}"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    return tmp_dir.resolve()


def _reload_cache_module(monkeypatch, cache_dir: Path):
    monkeypatch.setenv("PENRS_CACHE_DIR", str(cache_dir))
    importlib.reload(penrs_cache)
    return penrs_cache


def test_cache_key_is_deterministic(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / "cache")

    key1 = mod.cache_key("alpha", "MRNA", "earnings_call", "2025-01-10")
    key2 = mod.cache_key("alpha", "MRNA", "earnings_call", "2025-01-10")
    key3 = mod.cache_key("alpha", "MRNA", "earnings_call", "2025-01-11")

    assert key1 == key2
    assert key1 != key3
    assert len(key1) == 64


def test_cache_set_writes_json_with_metadata(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / "cache")
    payload = {"headline": "Sample", "score": 0.4}

    path = mod.cache_set(
        api="alpha",
        ticker="MRNA",
        doc_type="earnings_call",
        date="2025-01-10",
        payload=payload,
    )

    assert path.is_file()
    written = json.loads(path.read_text(encoding="utf-8"))

    assert "_cached_at" in written
    assert written["_api"] == "alpha"
    assert written["_ticker"] == "MRNA"
    assert written["_doc_type"] == "earnings_call"
    assert written["payload"] == payload


def test_cache_get_returns_payload_when_fresh(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / "cache")
    payload = {"k": "v", "n": 2}
    mod.cache_set(
        api="sec",
        ticker="BIIB",
        doc_type="10q",
        date="2025-02-01",
        payload=payload,
    )

    cached = mod.cache_get(
        api="sec",
        ticker="BIIB",
        doc_type="10q",
        date="2025-02-01",
        max_age_hours=12,
    )

    assert cached == payload


def test_cache_get_returns_none_for_missing_file(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / "cache")

    with patch.object(mod.logger, "info") as log_info:
        cached = mod.cache_get(
            api="ctgov",
            ticker="SAVA",
            doc_type="clinical_trial",
            date="2025-02-01",
            max_age_hours=24,
        )

    assert cached is None
    assert log_info.call_count >= 1


def test_cache_get_returns_none_when_expired(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / "cache")
    payload = {"x": 1}
    path = mod.cache_set(
        api="openfda",
        ticker="MRNA",
        doc_type="adverse_events",
        date="2025-02-01",
        payload=payload,
    )

    expired = json.loads(path.read_text(encoding="utf-8"))
    expired["_cached_at"] = (
        datetime.now(timezone.utc) - timedelta(hours=5)
    ).isoformat()
    path.write_text(json.dumps(expired, ensure_ascii=True), encoding="utf-8")

    with patch.object(mod.logger, "info") as log_info:
        cached = mod.cache_get(
            api="openfda",
            ticker="MRNA",
            doc_type="adverse_events",
            date="2025-02-01",
            max_age_hours=1,
        )

    assert cached is None
    assert log_info.call_count >= 1


def test_cache_operations_are_logged(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / f"cache-{uuid.uuid4().hex}")

    with patch.object(mod.logger, "info") as log_info:
        mod.cache_set(
            api="pubmed",
            ticker="SRPT",
            doc_type="publication",
            date="2025-03-02",
            payload={"paper_count": 3},
        )
        result = mod.cache_get(
            api="pubmed",
            ticker="SRPT",
            doc_type="publication",
            date="2025-03-02",
            max_age_hours=1,
        )

    assert result == {"paper_count": 3}
    assert log_info.call_count >= 3

## Pipeline Tests

In [ ]:
TEST_FILES_DIR = (Path.cwd() / "Test_files").resolve()


def local_tmp_dir() -> Path:
    TEST_FILES_DIR.mkdir(parents=True, exist_ok=True)
    tmp_dir = TEST_FILES_DIR / f"test_tmp_{uuid.uuid4().hex}"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    return tmp_dir.resolve()


class StubWorker:
    def __init__(
        self,
        *,
        name: str,
        weight: float,
        signal_density: float,
        score: float,
        fail: bool = False,
    ) -> None:
        self.name = name
        self.weight = weight
        self.signal_density = signal_density
        self.score = score
        self.fail = fail

    async def run(self, ticker: str, date_from: str, date_to: str) -> dict:
        if self.fail:
            raise RuntimeError(f"{self.name} exploded")
        return {
            "status": "available",
            "ticker": ticker,
            "date_from": date_from,
            "date_to": date_to,
            "worker": {
                "name": self.name,
                "weight": self.weight,
                "signal_density": self.signal_density,
            },
            "result": {
                "score": self.score,
                "summary": f"{self.name} summary",
            },
        }


class StubArbiter:
    def __init__(self) -> None:
        self.received: list[dict] | None = None

    def evaluate(self, worker_results: list[dict]) -> dict:
        self.received = list(worker_results)
        return {
            "status": "available",
            "worker_scores": [],
            "weighted_score": 0.33,
            "contradictions": [],
        }


class StubMaster:
    def __init__(self) -> None:
        self.received: dict | None = None

    async def synthesize(
        self,
        *,
        ticker: str,
        date_from: str,
        date_to: str,
        worker_results: list[dict],
        arbiter_result: dict,
    ) -> dict:
        self.received = {
            "ticker": ticker,
            "date_from": date_from,
            "date_to": date_to,
            "worker_results": worker_results,
            "arbiter_result": arbiter_result,
        }
        return {
            "status": "available",
            "final_score": arbiter_result.get("weighted_score", 0.0),
        }


def test_run_penrs_executes_pipeline_and_saves_report():
    working_dir = local_tmp_dir()
    original_cwd = Path.cwd()
    os.chdir(working_dir)
    workers = [
        StubWorker(name="W1", weight=1.0, signal_density=0.7, score=0.4),
        StubWorker(name="W2", weight=1.3, signal_density=0.8, score=0.2),
    ]
    arbiter = StubArbiter()
    master = StubMaster()

    try:
        report = asyncio.run(
            run_penrs(
                "MRNA",
                "2026-01-01",
                "2026-02-01",
                workers=workers,
                arbiter=arbiter,
                master=master,
            )
        )
    finally:
        os.chdir(original_cwd)

    assert arbiter.received is not None
    assert len(arbiter.received) == 2
    assert master.received is not None
    assert master.received["arbiter_result"]["weighted_score"] == 0.33

    report_path = Path(report["report_path"])
    assert report_path.exists()
    assert report_path.parent.name == "penrs_reports"

    saved_payload = json.loads(report_path.read_text(encoding="utf-8"))
    assert saved_payload["ticker"] == "MRNA"
    assert saved_payload["master"]["final_score"] == 0.33
    assert saved_payload["arbiter"]["status"] == "available"


def test_run_penrs_worker_failure_is_isolated():
    output_dir = local_tmp_dir()
    workers = [
        StubWorker(name="FailWorker", weight=1.0, signal_density=0.6, score=0.0, fail=True),
        StubWorker(name="GoodWorker", weight=1.0, signal_density=0.6, score=0.55),
    ]
    arbiter = StubArbiter()

    report = asyncio.run(
        run_penrs(
            "BIIB",
            "2026-01-01",
            "2026-02-01",
            workers=workers,
            arbiter=arbiter,
            report_dir=output_dir,
        )
    )

    statuses = {entry["worker"]["name"]: entry["status"] for entry in report["worker_results"]}
    assert statuses["FailWorker"] == "error"
    assert statuses["GoodWorker"] == "available"
    assert arbiter.received is not None
    assert len(arbiter.received) == 1
    assert arbiter.received[0]["worker"]["name"] == "GoodWorker"
    assert Path(report["report_path"]).exists()


def test_run_all_workers_uses_asyncio_gather_return_exceptions(monkeypatch):
    captured: dict[str, bool] = {}

    async def fake_gather(*coroutines, return_exceptions=False):
        captured["return_exceptions"] = return_exceptions
        results = []
        for coroutine in coroutines:
            try:
                results.append(await coroutine)
            except Exception as exc:
                if return_exceptions:
                    results.append(exc)
                else:
                    raise
        return results

    monkeypatch.setattr(asyncio, "gather", fake_gather)
    workers = [
        StubWorker(name="Broken", weight=1.0, signal_density=0.5, score=0.0, fail=True),
        StubWorker(name="Healthy", weight=1.0, signal_density=0.5, score=0.1),
    ]

    results = asyncio.run(run_all_workers(workers, "SRPT", "2026-01-01", "2026-02-01"))

    assert captured["return_exceptions"] is True
    statuses = {entry["worker"]["name"]: entry["status"] for entry in results}
    assert statuses["Broken"] == "error"
    assert statuses["Healthy"] == "available"

## Rate Limit Tests

In [ ]:
def _reload_rate_limit_module():
    importlib.reload(utils)
    utils._reset_rate_limit_state()
    return utils


def test_alpha_vantage_blocks_after_25_daily_calls(monkeypatch):
    mod = _reload_rate_limit_module()
    now = datetime(2026, 3, 2, 12, 0, tzinfo=timezone.utc)
    sleeps = []

    monkeypatch.setattr(mod, "_now_utc", lambda: now)
    monkeypatch.setattr(mod.time, "sleep", lambda seconds: sleeps.append(seconds))

    for _ in range(25):
        assert mod._check_rate_limit("alpha_vantage") is True
    assert mod._check_rate_limit("alpha_vantage") is False

    assert all(seconds == 12 for seconds in sleeps)


def test_alpha_vantage_sleeps_12_seconds_after_5_calls_same_minute(monkeypatch):
    mod = _reload_rate_limit_module()
    now = datetime(2026, 3, 2, 9, 30, tzinfo=timezone.utc)
    sleeps = []

    monkeypatch.setattr(mod, "_now_utc", lambda: now)
    monkeypatch.setattr(mod.time, "sleep", lambda seconds: sleeps.append(seconds))

    for _ in range(5):
        assert mod._check_rate_limit("alpha_vantage") is True
    assert mod._check_rate_limit("alpha_vantage") is True

    assert sleeps == [12]


def test_daily_counter_resets_at_midnight_boundary(monkeypatch):
    mod = _reload_rate_limit_module()
    previous_minute = datetime(2026, 3, 1, 23, 59, tzinfo=timezone.utc)
    midnight = datetime(2026, 3, 2, 0, 0, tzinfo=timezone.utc)

    mod._RATE_LIMIT_STATE["alpha_vantage"] = {
        "day_key": previous_minute.date().isoformat(),
        "daily_count": 25,
        "minute_key": previous_minute.replace(second=0, microsecond=0).isoformat(),
        "minute_count": 5,
    }

    monkeypatch.setattr(mod, "_now_utc", lambda: midnight)
    monkeypatch.setattr(mod.time, "sleep", lambda _seconds: None)

    assert mod._check_rate_limit("alpha_vantage") is True
    assert mod._RATE_LIMIT_STATE["alpha_vantage"]["daily_count"] == 1


def test_minute_counter_resets_on_new_minute_for_sec_edgar(monkeypatch):
    mod = _reload_rate_limit_module()
    now = datetime(2026, 3, 2, 11, 15, tzinfo=timezone.utc)
    next_minute = now + timedelta(minutes=1)

    monkeypatch.setattr(mod, "_now_utc", lambda: now)

    for _ in range(10):
        assert mod._check_rate_limit("sec_edgar") is True
    assert mod._check_rate_limit("sec_edgar") is False

    monkeypatch.setattr(mod, "_now_utc", lambda: next_minute)
    assert mod._check_rate_limit("sec_edgar") is True


def test_other_apis_use_configurable_rpm_limit(monkeypatch):
    mod = _reload_rate_limit_module()
    now = datetime(2026, 3, 2, 8, 45, tzinfo=timezone.utc)
    monkeypatch.setattr(mod, "_now_utc", lambda: now)

    assert mod._check_rate_limit("pubmed", rpm_limit=2) is True
    assert mod._check_rate_limit("pubmed", rpm_limit=2) is True
    assert mod._check_rate_limit("pubmed", rpm_limit=2) is False


def test_warning_logged_when_limits_approached_or_hit(monkeypatch):
    mod = _reload_rate_limit_module()
    now = datetime(2026, 3, 2, 8, 50, tzinfo=timezone.utc)
    monkeypatch.setattr(mod, "_now_utc", lambda: now)

    with patch.object(mod.logger, "warning") as log_warning:
        assert mod._check_rate_limit("pubmed", rpm_limit=2) is True
        assert mod._check_rate_limit("pubmed", rpm_limit=2) is True
        assert mod._check_rate_limit("pubmed", rpm_limit=2) is False

    assert log_warning.call_count >= 2

## Router Tests

In [6]:
def test_document_type_is_strict_enum_with_nine_values():
    assert issubclass(DocumentType, Enum)
    assert len(DocumentType) == 9
    assert set(DOCUMENT_API_ROUTING.keys()) == set(DocumentType)
    assert all(isinstance(api, str) and api for apis in DOCUMENT_API_ROUTING.values() for api in apis)


def test_penrs_fetch_document_requires_document_type_enum():
    with pytest.raises(TypeError):
        asyncio.run(
            penrs_fetch_document(
                ticker="MRNA",
                document_type="sec_10q",  # type: ignore[arg-type]
                date_range=("2026-01-01", "2026-02-01"),
                fetchers={},
            )
        )


def test_penrs_fetch_document_aggregates_multi_source_results():
    async def openfda_fetcher(ticker, date_range, document_type):
        assert ticker == "MRNA"
        assert date_range == ("2026-01-01", "2026-02-01")
        assert document_type is DocumentType.BIOMEDICAL_EVIDENCE
        return {"status": "available", "data": {"events": 3}}

    async def pubmed_fetcher(_ticker, _date_range, _document_type):
        return {"status": "available", "data": {"papers": 7}}

    result = asyncio.run(
        penrs_fetch_document(
            ticker="MRNA",
            document_type=DocumentType.BIOMEDICAL_EVIDENCE,
            date_range=("2026-01-01", "2026-02-01"),
            fetchers={
                "openfda": openfda_fetcher,
                "pubmed": pubmed_fetcher,
            },
        )
    )

    assert result["status"] == "available"
    assert result["data"]["apis_attempted"] == ["openfda", "pubmed"]
    assert result["data"]["sources"] == [
        {"api": "openfda", "data": {"events": 3}},
        {"api": "pubmed", "data": {"papers": 7}},
    ]


def test_penrs_fetch_document_returns_not_released_with_attempted_apis():
    async def sec_fetcher(_ticker, _date_range, _document_type):
        return {"status": "not_released"}

    result = asyncio.run(
        penrs_fetch_document(
            ticker="BIIB",
            document_type=DocumentType.SEC_10Q,
            date_range=("2026-01-01", "2026-02-01"),
            fetchers={"sec_edgar": sec_fetcher},
        )
    )

    assert result["status"] == "not_released"
    assert result["data"]["apis_attempted"] == ["sec_edgar"]


def test_penrs_fetch_document_handles_partial_failures():
    async def openfda_fetcher(_ticker, _date_range, _document_type):
        return {"status": "available", "data": {"events": 1}}

    async def pubmed_fetcher(_ticker, _date_range, _document_type):
        raise RuntimeError("pubmed outage")

    result = asyncio.run(
        penrs_fetch_document(
            ticker="SAVA",
            document_type=DocumentType.BIOMEDICAL_EVIDENCE,
            date_range=("2026-01-01", "2026-02-01"),
            fetchers={
                "openfda": openfda_fetcher,
                "pubmed": pubmed_fetcher,
            },
        )
    )

    assert result["status"] == "available"
    assert result["data"]["sources"] == [{"api": "openfda", "data": {"events": 1}}]
    assert result["data"]["partial_failures"] == [{"api": "pubmed", "error": "pubmed outage"}]

## Step 1 Tests

In [7]:
TEST_FILES_DIR = (Path.cwd() / "Test_files").resolve()


@pytest.fixture
def local_tmp_dir() -> Path:
    TEST_FILES_DIR.mkdir(parents=True, exist_ok=True)
    tmp_dir = TEST_FILES_DIR / f"test_tmp_{uuid.uuid4().hex}"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    return tmp_dir.resolve()


def test_env_defaults(local_tmp_dir, monkeypatch):
    monkeypatch.delenv("ALPHA_VANTAGE_API_KEY", raising=False)
    monkeypatch.delenv("PENRS_CACHE_DIR", raising=False)
    monkeypatch.delenv("PENRS_LOG_DIR", raising=False)
    monkeypatch.chdir(local_tmp_dir)

    # Patch at the source so that `from dotenv import load_dotenv`
    # during reload binds the mock, not the real function.
    with patch("dotenv.load_dotenv"):
        import penrs_mcp_server
        importlib.reload(penrs_mcp_server)

        assert penrs_mcp_server.ALPHA_VANTAGE_API_KEY == "demo"
        assert penrs_mcp_server.PENRS_CACHE_DIR == (local_tmp_dir / ".penrs_cache").resolve()
        assert penrs_mcp_server.PENRS_LOG_DIR == (local_tmp_dir / ".penrs_logs").resolve()


def test_dirs_created(local_tmp_dir, monkeypatch):
    monkeypatch.setenv("PENRS_CACHE_DIR", str(local_tmp_dir / "cache"))
    monkeypatch.setenv("PENRS_LOG_DIR", str(local_tmp_dir / "logs"))
    monkeypatch.chdir(local_tmp_dir)

    import penrs_mcp_server
    importlib.reload(penrs_mcp_server)

    assert (local_tmp_dir / "cache").is_dir()
    assert (local_tmp_dir / "logs").is_dir()


def test_api_key_from_env(monkeypatch, local_tmp_dir):
    monkeypatch.setenv("ALPHA_VANTAGE_API_KEY", "TESTKEY123")
    monkeypatch.setenv("PENRS_CACHE_DIR", str(local_tmp_dir / "cache"))
    monkeypatch.setenv("PENRS_LOG_DIR", str(local_tmp_dir / "logs"))
    monkeypatch.chdir(local_tmp_dir)

    import penrs_mcp_server
    importlib.reload(penrs_mcp_server)

    assert penrs_mcp_server.ALPHA_VANTAGE_API_KEY == "TESTKEY123"


def test_mcp_server_named(local_tmp_dir, monkeypatch):
    monkeypatch.setenv("PENRS_CACHE_DIR", str(local_tmp_dir / "cache"))
    monkeypatch.setenv("PENRS_LOG_DIR", str(local_tmp_dir / "logs"))
    monkeypatch.chdir(local_tmp_dir)

    import penrs_mcp_server
    importlib.reload(penrs_mcp_server)

    assert penrs_mcp_server.mcp.name == "penrs_mcp"

NameError: name 'pytest' is not defined

## Step 2 Tests

In [8]:
class DummyResponse:
    def __init__(
        self,
        status_code: int,
        *,
        json_data=None,
        text: str = "",
        json_error: Exception | None = None,
    ):
        self.status_code = status_code
        self._json_data = json_data
        self.text = text
        self._json_error = json_error

    @property
    def is_error(self) -> bool:
        return self.status_code >= 400

    def json(self):
        if self._json_error is not None:
            raise self._json_error
        return self._json_data


def _patch_async_client(monkeypatch, planned_results):
    instances = []

    class FakeAsyncClient:
        def __init__(self, timeout):
            self.timeout = timeout
            self.calls = []
            instances.append(self)

        async def __aenter__(self):
            return self

        async def __aexit__(self, exc_type, exc, tb):
            return False

        async def get(self, url, params=None, headers=None):
            self.calls.append({"url": url, "params": params, "headers": headers})
            if not planned_results:
                raise AssertionError("No planned response left for FakeAsyncClient.get()")

            next_result = planned_results.pop(0)
            if isinstance(next_result, Exception):
                raise next_result
            return next_result

    monkeypatch.setattr(penrs_http.httpx, "AsyncClient", FakeAsyncClient)
    return instances


def test_api_request_success_json_uses_default_timeout(monkeypatch):
    planned = [DummyResponse(200, json_data={"ok": True})]
    clients = _patch_async_client(monkeypatch, planned)

    result = asyncio.run(
        penrs_http._api_request(
            "https://example.test/data",
            params={"ticker": "MRNA"},
            headers={"X-Test": "1"},
            api_name="alpha_vantage",
        )
    )

    assert result == {"ok": True}
    assert len(clients) == 1
    assert clients[0].timeout == 30.0
    assert clients[0].calls == [
        {
            "url": "https://example.test/data",
            "params": {"ticker": "MRNA"},
            "headers": {"X-Test": "1"},
        }
    ]


def test_api_request_returns_text_when_json_parse_fails(monkeypatch):
    planned = [DummyResponse(200, text="raw text body", json_error=ValueError("bad json"))]
    _patch_async_client(monkeypatch, planned)

    result = asyncio.run(penrs_http._api_request("https://example.test/text"))

    assert result == {"text": "raw text body"}


def test_api_request_returns_structured_http_error_and_logs(monkeypatch):
    planned = [DummyResponse(500, text="server exploded")]
    _patch_async_client(monkeypatch, planned)

    with patch.object(penrs_http.logger, "error") as log_error:
        result = asyncio.run(penrs_http._api_request("https://example.test/fail", api_name="sec"))

    assert result == {"error": "HTTP 500", "detail": "server exploded"}
    assert log_error.call_count >= 1


def test_api_request_retries_429_503_with_exponential_backoff(monkeypatch):
    planned = [
        DummyResponse(429, text="rate limited"),
        DummyResponse(503, text="service unavailable"),
        DummyResponse(200, json_data={"status": "ok"}),
    ]
    clients = _patch_async_client(monkeypatch, planned)
    sleeps = []

    async def fake_sleep(seconds):
        sleeps.append(seconds)

    monkeypatch.setattr(penrs_http.asyncio, "sleep", fake_sleep)

    with patch.object(penrs_http.logger, "warning") as log_warning:
        result = asyncio.run(penrs_http._api_request("https://example.test/retry", api_name="ctgov"))

    assert result == {"status": "ok"}
    assert sleeps == [1, 2]
    assert len(clients[0].calls) == 3
    assert log_warning.call_count == 2


def test_api_request_stops_after_max_retries_on_429(monkeypatch):
    planned = [
        DummyResponse(429, text="limited"),
        DummyResponse(429, text="limited"),
        DummyResponse(429, text="limited"),
        DummyResponse(429, text="still limited"),
    ]
    clients = _patch_async_client(monkeypatch, planned)
    sleeps = []

    async def fake_sleep(seconds):
        sleeps.append(seconds)

    monkeypatch.setattr(penrs_http.asyncio, "sleep", fake_sleep)

    result = asyncio.run(penrs_http._api_request("https://example.test/limit", api_name="alpha"))

    assert result == {"error": "HTTP 429", "detail": "still limited"}
    assert len(clients[0].calls) == 4
    assert sleeps == [1, 2, 4]


def test_api_request_timeout_is_user_friendly_and_logged(monkeypatch):
    planned = [httpx.TimeoutException("took too long")]
    _patch_async_client(monkeypatch, planned)

    with patch.object(penrs_http.logger, "error") as log_error:
        result = asyncio.run(penrs_http._api_request("https://example.test/timeout", api_name="openfda"))

    assert result["error"] == "Request timed out"
    assert "https://example.test/timeout" in result["detail"]
    assert log_error.call_count >= 1


def test_api_request_request_error_is_user_friendly_and_logged(monkeypatch):
    request = httpx.Request("GET", "https://example.test/network")
    planned = [httpx.RequestError("network unreachable", request=request)]
    _patch_async_client(monkeypatch, planned)

    with patch.object(penrs_http.logger, "error") as log_error:
        result = asyncio.run(penrs_http._api_request("https://example.test/network", api_name="pubmed"))

    assert result["error"] == "Request failed"
    assert "network unreachable" in result["detail"]
    assert log_error.call_count >= 1


def test_api_request_respects_custom_timeout(monkeypatch):
    planned = [DummyResponse(200, json_data={"ok": True})]
    clients = _patch_async_client(monkeypatch, planned)

    result = asyncio.run(
        penrs_http._api_request(
            "https://example.test/custom-timeout",
            timeout=5.5,
            api_name="alpha_vantage",
        )
    )

    assert result == {"ok": True}
    assert clients[0].timeout == 5.5

## Worker Tests

In [9]:
def test_truncate_for_context_preserves_start_and_end():
    source = "A" * 80 + "MIDDLE" + "Z" * 80
    truncated = truncate_for_context(source, max_chars=60)

    assert len(truncated) == 60
    assert truncated.startswith("A")
    assert truncated.endswith("Z")
    assert "[truncated" in truncated


def test_parse_json_response_handles_markdown_block():
    worker = PENRSWorker(
        name="TestWorker",
        weight=1.0,
        signal_density=0.5,
        rubric_id="worker_test",
        document_type=DocumentType.SEC_10Q,
    )
    response = "analysis:\n```json\n{\"score\": 0.7, \"summary\": \"ok\"}\n```\nextra"

    parsed = worker.parse_json_response(response)

    assert parsed == {"score": 0.7, "summary": "ok"}


def test_parse_json_response_handles_embedded_prose_and_invalid_json_fallback():
    worker = PENRSWorker(
        name="TestWorker",
        weight=1.0,
        signal_density=0.5,
        rubric_id="worker_test",
        document_type=DocumentType.SEC_10Q,
    )
    prose_response = 'Before note. {"signal": "bearish", "confidence": 0.81} After note.'
    invalid_response = "I could not determine anything with confidence."

    parsed_from_prose = worker.parse_json_response(prose_response)
    parsed_invalid = worker.parse_json_response(invalid_response)

    assert parsed_from_prose == {"signal": "bearish", "confidence": 0.81}
    assert parsed_invalid["parse_error"] == "unable_to_parse_json"
    assert parsed_invalid["raw_response"] == invalid_response


def test_run_fetches_rubric_and_document_and_enriches_metadata():
    calls = {"rubric": 0, "document": 0, "prompt": None}

    async def fake_rubric_fetcher(rubric_id):
        calls["rubric"] += 1
        assert rubric_id == "worker_earnings"
        return {"name": "Earnings rubric", "threshold": 0.6}

    async def fake_document_fetcher(ticker, document_type, date_range):
        calls["document"] += 1
        assert ticker == "MRNA"
        assert document_type is DocumentType.EARNINGS_CALL
        assert date_range == {"from": "2026-01-01", "to": "2026-02-01"}
        return {
            "status": "available",
            "data": {
                "ticker": ticker,
                "sources": [{"api": "alpha_vantage", "data": {"transcript": "A" * 300}}],
            },
        }

    async def fake_llm(prompt):
        calls["prompt"] = prompt
        return "```json\n{\"score\": 0.42, \"thesis\": \"mixed\"}\n```"

    worker = PENRSWorker(
        name="EarningsWorker",
        weight=1.2,
        signal_density=0.75,
        rubric_id="worker_earnings",
        document_type=DocumentType.EARNINGS_CALL,
        rubric_fetcher=fake_rubric_fetcher,
        document_fetcher=fake_document_fetcher,
        llm_invoker=fake_llm,
        max_context_chars=120,
    )

    result = asyncio.run(worker.run("MRNA", "2026-01-01", "2026-02-01"))

    assert calls["rubric"] == 1
    assert calls["document"] == 1
    assert "Document excerpt:" in calls["prompt"]
    assert result["status"] == "available"
    assert result["worker"] == {
        "name": "EarningsWorker",
        "weight": 1.2,
        "signal_density": 0.75,
    }
    assert result["result"] == {"score": 0.42, "thesis": "mixed"}


def test_run_handles_not_released_without_llm_call():
    calls = {"llm": 0}

    async def fake_document_fetcher(_ticker, _document_type, _date_range):
        return {
            "status": "not_released",
            "data": {
                "apis_attempted": ["sec_edgar"],
                "reason": "Filing not yet published",
            },
        }

    async def fake_llm(_prompt):
        calls["llm"] += 1
        return {"score": 0.0}

    worker = PENRSWorker(
        name="SECFilingWorker",
        weight=1.0,
        signal_density=0.4,
        rubric_id="worker_sec",
        document_type=DocumentType.SEC_10Q,
        rubric_fetcher=lambda _rubric_id: {"stub": True},
        document_fetcher=fake_document_fetcher,
        llm_invoker=fake_llm,
    )

    result = asyncio.run(worker.run("BIIB", "2026-01-01", "2026-02-01"))

    assert result["status"] == "not_released"
    assert result["apis_attempted"] == ["sec_edgar"]
    assert result["worker"]["name"] == "SECFilingWorker"
    assert calls["llm"] == 0

## Worker Spec 2 Unit Tests

In [ ]:
def test_spec_2_1_build_prompt_injects_strict_schema_and_verbatim_constraint():
    mod = import_worker_nodes(force_reload=True)
    worker = mod.PENRSWorker(
        name="Clinical Signals",
        weight=1.0,
        signal_density=0.9,
        rubric_id="test_rubric",
        document_type=DocumentType.NEWS_SENTIMENT,
    )

    prompt = worker.build_prompt(
        ticker="MRNA",
        date_from="2026-01-01",
        date_to="2026-02-01",
        rubric={"criteria": "test"},
        document_excerpt="Phase 3 enrollment delay was disclosed.",
    )

    assert "Return only JSON." in prompt
    assert "You MUST return a strictly valid JSON object adhering to this exact schema:" in prompt
    assert '"score": <float between -1.0 and 1.0>' in prompt
    assert '"thesis": <string>' in prompt
    assert '"evidence_nodes": [' in prompt
    assert '{"verbatim_quote": <exact substring from document>, "reasoning": <string>}' in prompt
    assert (
        "The 'verbatim_quote' MUST be an exact, character-for-character substring of the provided Document excerpt."
        in prompt
    )


def test_spec_2_2_parse_json_response_defaults_when_keys_missing():
    mod = import_worker_nodes(force_reload=True)
    worker = mod.PENRSWorker(
        name="Clinical Signals",
        weight=1.0,
        signal_density=0.9,
        rubric_id="test_rubric",
        document_type=DocumentType.NEWS_SENTIMENT,
    )

    parsed = worker.parse_json_response("{}")

    assert parsed == {"score": 0.0, "thesis": "Parse failure", "evidence_nodes": []}


def test_spec_2_2_parse_json_response_defaults_when_any_required_key_missing():
    mod = import_worker_nodes(force_reload=True)
    worker = mod.PENRSWorker(
        name="Clinical Signals",
        weight=1.0,
        signal_density=0.9,
        rubric_id="test_rubric",
        document_type=DocumentType.NEWS_SENTIMENT,
    )
    default_payload = {"score": 0.0, "thesis": "Parse failure", "evidence_nodes": []}

    missing_score = worker.parse_json_response(json.dumps({"thesis": "x", "evidence_nodes": []}))
    missing_thesis = worker.parse_json_response(json.dumps({"score": 0.2, "evidence_nodes": []}))
    missing_evidence = worker.parse_json_response(json.dumps({"score": 0.2, "thesis": "x"}))

    assert missing_score == default_payload
    assert missing_thesis == default_payload
    assert missing_evidence == default_payload


def test_spec_2_2_parse_json_response_enforces_types_and_clamps_score():
    mod = import_worker_nodes(force_reload=True)
    worker = mod.PENRSWorker(
        name="Clinical Signals",
        weight=1.0,
        signal_density=0.9,
        rubric_id="test_rubric",
        document_type=DocumentType.NEWS_SENTIMENT,
    )

    parsed = worker.parse_json_response(
        json.dumps(
            {
                "score": "7.5",
                "thesis": 999,
                "evidence_nodes": {"invalid": True},
            }
        )
    )

    assert parsed["score"] == 1.0
    assert parsed["thesis"] == "Parse failure"
    assert parsed["evidence_nodes"] == []


def test_spec_2_2_parse_json_response_coerces_score_to_float_type():
    mod = import_worker_nodes(force_reload=True)
    worker = mod.PENRSWorker(
        name="Clinical Signals",
        weight=1.0,
        signal_density=0.9,
        rubric_id="test_rubric",
        document_type=DocumentType.NEWS_SENTIMENT,
    )

    parsed = worker.parse_json_response(
        json.dumps(
            {
                "score": 1,
                "thesis": "typed value",
                "evidence_nodes": [],
            }
        )
    )

    assert isinstance(parsed["score"], float)
    assert parsed["score"] == 1.0


def test_spec_2_2_parse_json_response_supports_embedded_and_fenced_json():
    mod = import_worker_nodes(force_reload=True)
    worker = mod.PENRSWorker(
        name="Clinical Signals",
        weight=1.0,
        signal_density=0.9,
        rubric_id="test_rubric",
        document_type=DocumentType.NEWS_SENTIMENT,
    )
    expected = {
        "score": -0.25,
        "thesis": "Delay risk increasing.",
        "evidence_nodes": [{"verbatim_quote": "delaying our phase 3 trials", "reasoning": "Direct disclosure."}],
    }

    parsed_fenced = worker.parse_json_response(f"```json\n{json.dumps(expected)}\n```")
    parsed_embedded = worker.parse_json_response(f"Model output: {json.dumps(expected)} trailing notes")

    assert parsed_fenced == expected
    assert parsed_embedded == expected


def test_spec_2_2_parse_json_response_returns_default_for_non_json():
    mod = import_worker_nodes(force_reload=True)
    worker = mod.PENRSWorker(
        name="Clinical Signals",
        weight=1.0,
        signal_density=0.9,
        rubric_id="test_rubric",
        document_type=DocumentType.NEWS_SENTIMENT,
    )

    parsed = worker.parse_json_response("not-json-at-all")

    assert parsed == {"score": 0.0, "thesis": "Parse failure", "evidence_nodes": []}


def test_spec_2_2_parse_json_response_handles_dict_and_none_inputs():
    mod = import_worker_nodes(force_reload=True)
    worker = mod.PENRSWorker(
        name="Clinical Signals",
        weight=1.0,
        signal_density=0.9,
        rubric_id="test_rubric",
        document_type=DocumentType.NEWS_SENTIMENT,
    )

    parsed_dict = worker.parse_json_response(
        {"score": -0.4, "thesis": "dict-input", "evidence_nodes": []}
    )
    parsed_none = worker.parse_json_response(None)

    assert parsed_dict == {"score": -0.4, "thesis": "dict-input", "evidence_nodes": []}
    assert parsed_none == {"score": 0.0, "thesis": "Parse failure", "evidence_nodes": []}


## Worker Spec 2 Integration Tests

In [ ]:
def test_spec_2_3_run_passes_anthropic_system_prompt_and_parses_response():
    async def _run():
        mod = import_worker_nodes(force_reload=True)
        seen = {"prompt": None, "system": None}
        llm_payload = {
            "score": -0.9,
            "thesis": "Management is bailing out while pushing dilute and delay tactics.",
            "evidence_nodes": [
                {
                    "verbatim_quote": "we are severely delaying our phase 3 trials",
                    "reasoning": "Direct delay admission.",
                }
            ],
        }

        async def fake_document_fetcher(_ticker, _doc_type, _date_range):
            return {
                "status": "available",
                "data": "Q3 earnings were solid, however, we are severely delaying our phase 3 trials due to enrollment issues.",
            }

        def fake_rubric_fetcher(_rubric_id):
            return {"criteria": "test"}

        def fake_llm_invoker(prompt, *, system):
            seen["prompt"] = prompt
            seen["system"] = system
            return json.dumps(llm_payload)

        worker = mod.PENRSWorker(
            name="Dilute and Delay",
            weight=1.0,
            signal_density=1.0,
            rubric_id="test_rubric",
            document_type=DocumentType.NEWS_SENTIMENT,
            rubric_fetcher=fake_rubric_fetcher,
            document_fetcher=fake_document_fetcher,
            llm_invoker=fake_llm_invoker,
        )

        result = await worker.run("TEST", "2026-01-01", "2026-02-01")

        assert result["status"] == "available"
        assert seen["system"] == "Respond only in valid JSON."
        assert "You MUST return a strictly valid JSON object adhering to this exact schema:" in seen["prompt"]
        assert result["result"] == llm_payload

    asyncio.run(_run())


def test_spec_2_3_run_falls_back_when_invoker_rejects_system_kwarg():
    async def _run():
        mod = import_worker_nodes(force_reload=True)
        calls = {"count": 0}

        def fake_rubric_fetcher(_rubric_id):
            return {"criteria": "fallback-test"}

        async def fake_document_fetcher(_ticker, _doc_type, _date_range):
            return {"status": "available", "data": "Simple excerpt"}

        def positional_only_llm(prompt):
            calls["count"] += 1
            return '{"score": 0.1, "thesis": "ok", "evidence_nodes": []}'

        worker = mod.PENRSWorker(
            name="Fallback Worker",
            weight=1.0,
            signal_density=1.0,
            rubric_id="test_rubric",
            document_type=DocumentType.NEWS_SENTIMENT,
            rubric_fetcher=fake_rubric_fetcher,
            document_fetcher=fake_document_fetcher,
            llm_invoker=positional_only_llm,
        )

        result = await worker.run("TEST", "2026-01-01", "2026-02-01")

        assert calls["count"] == 1
        assert result["status"] == "available"
        assert result["result"]["score"] == 0.1
        assert result["result"]["thesis"] == "ok"
        assert result["result"]["evidence_nodes"] == []

    asyncio.run(_run())


def test_spec_2_3_default_llm_invoker_accepts_system_and_returns_schema_safe_payload():
    async def _run():
        mod = import_worker_nodes(force_reload=True)

        def fake_rubric_fetcher(_rubric_id):
            return {"criteria": "default-llm-test"}

        async def fake_document_fetcher(_ticker, _doc_type, _date_range):
            return {"status": "available", "data": "Document text"}

        worker = mod.PENRSWorker(
            name="Default Invoker",
            weight=1.0,
            signal_density=1.0,
            rubric_id="test_rubric",
            document_type=DocumentType.NEWS_SENTIMENT,
            rubric_fetcher=fake_rubric_fetcher,
            document_fetcher=fake_document_fetcher,
            llm_invoker=None,
        )

        result = await worker.run("TEST", "2026-01-01", "2026-02-01")

        assert result["status"] == "available"
        assert result["result"] == {"score": 0.0, "thesis": "Parse failure", "evidence_nodes": []}

    asyncio.run(_run())


def test_spec_2_3_run_passes_system_prompt_to_async_kwargs_invoker():
    async def _run():
        mod = import_worker_nodes(force_reload=True)
        seen = {"system": None}

        def fake_rubric_fetcher(_rubric_id):
            return {"criteria": "kwargs-test"}

        async def fake_document_fetcher(_ticker, _doc_type, _date_range):
            return {"status": "available", "data": "Document excerpt for kwargs invoker."}

        async def async_kwargs_llm(prompt, **kwargs):
            _ = prompt
            seen["system"] = kwargs.get("system")
            return '{"score": -0.2, "thesis": "ok", "evidence_nodes": []}'

        worker = mod.PENRSWorker(
            name="Kwargs Worker",
            weight=1.0,
            signal_density=1.0,
            rubric_id="test_rubric",
            document_type=DocumentType.NEWS_SENTIMENT,
            rubric_fetcher=fake_rubric_fetcher,
            document_fetcher=fake_document_fetcher,
            llm_invoker=async_kwargs_llm,
        )

        result = await worker.run("TEST", "2026-01-01", "2026-02-01")

        assert seen["system"] == "Respond only in valid JSON."
        assert result["status"] == "available"
        assert result["result"]["score"] == -0.2

    asyncio.run(_run())


## Fetcher Unit Tests

In [ ]:
from unittest.mock import AsyncMock, Mock


def test_spec_1_1_required_imports_and_mcp_tool_registration_present():
    server = import_penrs_server(force_reload=True)
    assert hasattr(server, "os")
    assert hasattr(server, "logging")
    assert hasattr(server, "FastMCP")
    assert hasattr(server, "_api_request")
    assert hasattr(server, "cache_set")
    assert hasattr(server, "PENRS_CACHE_DIR")
    assert server.mcp.name == "penrs_mcp"
    for tool_name in ("fetch_alpha_vantage", "fetch_sec_edgar", "fetch_openfda", "fetch_pubmed"):
        assert getattr(server, tool_name)._is_mcp_tool


def test_spec_1_2_alpha_vantage_success_routes_and_caches():
    async def _run():
        server = import_penrs_server(force_reload=True)
        result_payload = {"ok": True}
        with patch.object(server, "_api_request", AsyncMock(return_value=result_payload)) as api_mock, \
             patch.object(server, "cache_set", Mock()) as cache_mock:
            result = await server.fetch_alpha_vantage("MRNA", "TIME_SERIES_DAILY", "2026-01-01")
        assert result == result_payload
        api_mock.assert_awaited_once_with(
            "https://www.alphavantage.co/query",
            params={"function": "TIME_SERIES_DAILY", "symbol": "MRNA", "apikey": server.ALPHA_VANTAGE_API_KEY},
            api_name="alpha_vantage",
        )
        cache_mock.assert_called_once_with(
            api="alpha_vantage", ticker="MRNA", doc_type="TIME_SERIES_DAILY",
            date="2026-01-01", payload=result_payload,
        )
    asyncio.run(_run())


def test_spec_1_2_alpha_vantage_error_skips_cache():
    async def _run():
        server = import_penrs_server(force_reload=True)
        error_payload = {"error": "upstream failure"}
        with patch.object(server, "_api_request", AsyncMock(return_value=error_payload)) as api_mock, \
             patch.object(server, "cache_set", Mock()) as cache_mock:
            result = await server.fetch_alpha_vantage("MRNA", "TIME_SERIES_DAILY")
        assert result == error_payload
        api_mock.assert_awaited_once()
        cache_mock.assert_not_called()
    asyncio.run(_run())


def test_spec_1_3_sec_edgar_success_routes_headers_and_caches():
    async def _run():
        server = import_penrs_server(force_reload=True)
        result_payload = {"filing": "raw text"}
        with patch.object(server, "_api_request", AsyncMock(return_value=result_payload)) as api_mock, \
             patch.object(server, "cache_set", Mock()) as cache_mock:
            result = await server.fetch_sec_edgar("MRNA", "0000123456", "10q.htm")
        assert result == result_payload
        api_mock.assert_awaited_once_with(
            "https://www.sec.gov/Archives/edgar/data/MRNA/0000123456/10q.htm",
            headers={"User-Agent": server.SEC_USER_AGENT},
            api_name="sec_edgar",
        )
        cache_mock.assert_called_once_with(
            api="sec_edgar", ticker="MRNA", doc_type="filing", date=None, payload=result_payload,
        )
    asyncio.run(_run())


def test_spec_1_3_sec_edgar_error_skips_cache():
    async def _run():
        server = import_penrs_server(force_reload=True)
        error_payload = {"error": "forbidden"}
        with patch.object(server, "_api_request", AsyncMock(return_value=error_payload)), \
             patch.object(server, "cache_set", Mock()) as cache_mock:
            result = await server.fetch_sec_edgar("MRNA", "0000123456", "10q.htm")
        assert result == error_payload
        cache_mock.assert_not_called()
    asyncio.run(_run())


def test_spec_1_4_openfda_without_api_key_routes_and_caches():
    async def _run():
        server = import_penrs_server(force_reload=True)
        result_payload = {"results": []}
        with patch.object(server, "OPENFDA_API_KEY", None), \
             patch.object(server, "_api_request", AsyncMock(return_value=result_payload)) as api_mock, \
             patch.object(server, "cache_set", Mock()) as cache_mock:
            result = await server.fetch_openfda("MRNA", limit=7)
        assert result == result_payload
        api_mock.assert_awaited_once_with(
            "https://api.fda.gov/drug/event.json",
            params={"search": "patient.drug.medicinalproduct:MRNA", "limit": 7},
            api_name="openfda",
        )
        cache_mock.assert_called_once_with(
            api="openfda", ticker="MRNA", doc_type="adverse_events", date=None, payload=result_payload,
        )
    asyncio.run(_run())


def test_spec_1_4_openfda_includes_api_key_when_present():
    async def _run():
        server = import_penrs_server(force_reload=True)
        result_payload = {"results": []}
        with patch.object(server, "OPENFDA_API_KEY", "openfda-key"), \
             patch.object(server, "_api_request", AsyncMock(return_value=result_payload)) as api_mock:
            await server.fetch_openfda("BIIB")
        assert api_mock.await_args.kwargs["params"]["api_key"] == "openfda-key"
        assert api_mock.await_args.kwargs["params"]["limit"] == 10
    asyncio.run(_run())


def test_spec_1_4_openfda_error_skips_cache():
    async def _run():
        server = import_penrs_server(force_reload=True)
        error_payload = {"error": "upstream"}
        with patch.object(server, "_api_request", AsyncMock(return_value=error_payload)), \
             patch.object(server, "cache_set", Mock()) as cache_mock:
            result = await server.fetch_openfda("MRNA")
        assert result == error_payload
        cache_mock.assert_not_called()
    asyncio.run(_run())


def test_spec_1_5_pubmed_without_api_key_routes_and_caches():
    async def _run():
        server = import_penrs_server(force_reload=True)
        result_payload = {"esearchresult": {"idlist": ["1", "2"]}}
        with patch.object(server, "NCBI_API_KEY", None), \
             patch.object(server, "_api_request", AsyncMock(return_value=result_payload)) as api_mock, \
             patch.object(server, "cache_set", Mock()) as cache_mock:
            result = await server.fetch_pubmed("multiple sclerosis", retmax=2)
        assert result == result_payload
        api_mock.assert_awaited_once_with(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params={"db": "pubmed", "term": "multiple sclerosis", "retmode": "json", "retmax": 2},
            api_name="pubmed",
        )
        cache_mock.assert_called_once_with(
            api="pubmed", ticker="multiple_sclerosis", doc_type="publications",
            date=None, payload=result_payload,
        )
    asyncio.run(_run())


def test_spec_1_5_pubmed_includes_api_key_when_present():
    async def _run():
        server = import_penrs_server(force_reload=True)
        result_payload = {"esearchresult": {"idlist": []}}
        with patch.object(server, "NCBI_API_KEY", "ncbi-key"), \
             patch.object(server, "_api_request", AsyncMock(return_value=result_payload)) as api_mock:
            await server.fetch_pubmed("glioblastoma")
        assert api_mock.await_args.kwargs["params"]["api_key"] == "ncbi-key"
        assert api_mock.await_args.kwargs["params"]["retmax"] == 5
    asyncio.run(_run())


def test_spec_1_5_pubmed_error_skips_cache():
    async def _run():
        server = import_penrs_server(force_reload=True)
        error_payload = {"error": "failure"}
        with patch.object(server, "_api_request", AsyncMock(return_value=error_payload)), \
             patch.object(server, "cache_set", Mock()) as cache_mock:
            result = await server.fetch_pubmed("glioblastoma")
        assert result == error_payload
        cache_mock.assert_not_called()
    asyncio.run(_run())

## Fetcher Cache Integration Tests

In [ ]:
import hashlib
import shutil
import tempfile


def _fetcher_load_cached_record(*, server, api, ticker, doc_type, date):
    material = f"{api}|{ticker}|{doc_type}|{date or ''}"
    cache_key = hashlib.sha256(material.encode("utf-8")).hexdigest()
    path = server.PENRS_CACHE_DIR / f"{cache_key}.json"
    assert path.exists(), f"Expected cache file to exist: {path}"
    return json.loads(path.read_text(encoding="utf-8"))


def test_spec_1_2_alpha_vantage_success_writes_immutable_ground_truth_to_disk():
    tmpdir = Path(tempfile.mkdtemp(prefix="penrs_fetcher_integration_")).resolve()
    try:
        async def _run():
            with patch.dict(os.environ, {"PENRS_CACHE_DIR": str(tmpdir / "cache")}, clear=False):
                server = import_penrs_server(force_reload=True)
            payload = {"series": [{"close": 100.0}]}
            with patch.object(server, "_api_request", AsyncMock(return_value=payload)):
                result = await server.fetch_alpha_vantage("MRNA", "TIME_SERIES_DAILY", "2026-01-01")
            assert result == payload
            record = _fetcher_load_cached_record(
                server=server, api="alpha_vantage", ticker="MRNA",
                doc_type="TIME_SERIES_DAILY", date="2026-01-01",
            )
            assert record["_api"] == "alpha_vantage"
            assert record["_ticker"] == "MRNA"
            assert record["_doc_type"] == "TIME_SERIES_DAILY"
            assert record["_date"] == "2026-01-01"
            assert record["payload"] == payload
            assert "_cached_at" in record
        asyncio.run(_run())
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def test_spec_1_3_sec_edgar_success_writes_expected_cache_record():
    tmpdir = Path(tempfile.mkdtemp(prefix="penrs_fetcher_integration_")).resolve()
    try:
        async def _run():
            with patch.dict(os.environ, {"PENRS_CACHE_DIR": str(tmpdir / "cache")}, clear=False):
                server = import_penrs_server(force_reload=True)
            payload = {"filing": "<html>10Q</html>"}
            with patch.object(server, "_api_request", AsyncMock(return_value=payload)):
                result = await server.fetch_sec_edgar("MRNA", "0000123456", "10q.htm")
            assert result == payload
            record = _fetcher_load_cached_record(
                server=server, api="sec_edgar", ticker="MRNA", doc_type="filing", date=None,
            )
            assert record["_api"] == "sec_edgar"
            assert record["_ticker"] == "MRNA"
            assert record["_doc_type"] == "filing"
            assert record["_date"] is None
            assert record["payload"] == payload
        asyncio.run(_run())
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def test_spec_1_4_openfda_success_writes_expected_cache_record():
    tmpdir = Path(tempfile.mkdtemp(prefix="penrs_fetcher_integration_")).resolve()
    try:
        async def _run():
            with patch.dict(os.environ, {"PENRS_CACHE_DIR": str(tmpdir / "cache")}, clear=False):
                server = import_penrs_server(force_reload=True)
            payload = {"results": [{"id": "abc"}]}
            with patch.object(server, "_api_request", AsyncMock(return_value=payload)):
                result = await server.fetch_openfda("BIIB", limit=3)
            assert result == payload
            record = _fetcher_load_cached_record(
                server=server, api="openfda", ticker="BIIB", doc_type="adverse_events", date=None,
            )
            assert record["_api"] == "openfda"
            assert record["_ticker"] == "BIIB"
            assert record["_doc_type"] == "adverse_events"
            assert record["payload"] == payload
        asyncio.run(_run())
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def test_spec_1_5_pubmed_success_writes_normalized_cache_record():
    tmpdir = Path(tempfile.mkdtemp(prefix="penrs_fetcher_integration_")).resolve()
    try:
        async def _run():
            with patch.dict(os.environ, {"PENRS_CACHE_DIR": str(tmpdir / "cache")}, clear=False):
                server = import_penrs_server(force_reload=True)
            payload = {"esearchresult": {"idlist": ["1"]}}
            with patch.object(server, "_api_request", AsyncMock(return_value=payload)):
                result = await server.fetch_pubmed("multiple sclerosis", retmax=1)
            assert result == payload
            record = _fetcher_load_cached_record(
                server=server, api="pubmed", ticker="multiple_sclerosis",
                doc_type="publications", date=None,
            )
            assert record["_api"] == "pubmed"
            assert record["_ticker"] == "multiple_sclerosis"
            assert record["_doc_type"] == "publications"
            assert record["payload"] == payload
        asyncio.run(_run())
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def test_error_responses_do_not_write_cache_files():
    tmpdir = Path(tempfile.mkdtemp(prefix="penrs_fetcher_integration_")).resolve()
    try:
        async def _run():
            with patch.dict(os.environ, {"PENRS_CACHE_DIR": str(tmpdir / "cache")}, clear=False):
                server = import_penrs_server(force_reload=True)
            with patch.object(server, "_api_request", AsyncMock(return_value={"error": "upstream"})):
                result = await server.fetch_openfda("MRNA", limit=2)
            assert result == {"error": "upstream"}
            assert not any(server.PENRS_CACHE_DIR.glob("*.json"))
        asyncio.run(_run())
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

## Spec 3 Worker Hallucination Unit Tests

In [ ]:
SYSTEM_NOTE = "[SYSTEM NOTE: Score neutralized due to hallucinated evidence.]"


def test_spec_3_1_prunes_hallucinated_nodes_and_keeps_valid_quotes():
    async def _run():
        mod = import_worker_nodes(force_reload=True)

        def fake_rubric_fetcher(_rubric_id):
            return {"criteria": "quote-validation"}

        async def fake_document_fetcher(_ticker, _doc_type, _date_range):
            return {
                "status": "available",
                "data": "Revenue grew sequentially and guidance was reaffirmed for FY26.",
            }

        llm_payload = {
            "score": 0.6,
            "thesis": "Mostly constructive update.",
            "evidence_nodes": [
                {"verbatim_quote": "guidance was reaffirmed", "reasoning": "Directly in excerpt."},
                {"verbatim_quote": "pipeline discontinued", "reasoning": "Hallucinated."},
                {"reasoning": "Missing quote key."},
                "not-a-dict",
            ],
        }

        def fake_llm_invoker(_prompt, *, system):
            _ = system
            return json.dumps(llm_payload)

        worker = mod.PENRSWorker(
            name="Validation Worker",
            weight=1.0,
            signal_density=1.0,
            rubric_id="test_rubric",
            document_type=DocumentType.NEWS_SENTIMENT,
            rubric_fetcher=fake_rubric_fetcher,
            document_fetcher=fake_document_fetcher,
            llm_invoker=fake_llm_invoker,
        )

        result = await worker.run("TEST", "2026-01-01", "2026-02-01")

        assert result["result"]["score"] == 0.6
        assert result["result"]["evidence_nodes"] == [
            {"verbatim_quote": "guidance was reaffirmed", "reasoning": "Directly in excerpt."}
        ]
        assert SYSTEM_NOTE not in result["result"]["thesis"]

    asyncio.run(_run())


def test_spec_3_1_neutralizes_score_and_appends_system_note_when_all_nodes_are_hallucinated():
    async def _run():
        mod = import_worker_nodes(force_reload=True)

        def fake_rubric_fetcher(_rubric_id):
            return {"criteria": "neutralize"}

        async def fake_document_fetcher(_ticker, _doc_type, _date_range):
            return {"status": "available", "data": "Only this sentence is real."}

        def fake_llm_invoker(_prompt, *, system):
            _ = system
            return json.dumps(
                {
                    "score": -0.8,
                    "thesis": "High conviction downside.",
                    "evidence_nodes": [
                        {"verbatim_quote": "fabricated phrase", "reasoning": "Not present."},
                    ],
                }
            )

        worker = mod.PENRSWorker(
            name="Neutralize Worker",
            weight=1.0,
            signal_density=1.0,
            rubric_id="test_rubric",
            document_type=DocumentType.NEWS_SENTIMENT,
            rubric_fetcher=fake_rubric_fetcher,
            document_fetcher=fake_document_fetcher,
            llm_invoker=fake_llm_invoker,
        )

        result = await worker.run("TEST", "2026-01-01", "2026-02-01")

        assert result["result"]["evidence_nodes"] == []
        assert result["result"]["score"] == 0.0
        assert SYSTEM_NOTE in result["result"]["thesis"]

    asyncio.run(_run())


def test_spec_3_1_system_note_is_not_duplicated_when_already_present():
    async def _run():
        mod = import_worker_nodes(force_reload=True)

        def fake_rubric_fetcher(_rubric_id):
            return {"criteria": "dedupe-note"}

        async def fake_document_fetcher(_ticker, _doc_type, _date_range):
            return {"status": "available", "data": "Ground-truth excerpt."}

        def fake_llm_invoker(_prompt, *, system):
            _ = system
            return json.dumps(
                {
                    "score": 0.4,
                    "thesis": f"Original thesis. {SYSTEM_NOTE}",
                    "evidence_nodes": [{"verbatim_quote": "not in excerpt", "reasoning": "Bad quote."}],
                }
            )

        worker = mod.PENRSWorker(
            name="No Duplicate Note",
            weight=1.0,
            signal_density=1.0,
            rubric_id="test_rubric",
            document_type=DocumentType.NEWS_SENTIMENT,
            rubric_fetcher=fake_rubric_fetcher,
            document_fetcher=fake_document_fetcher,
            llm_invoker=fake_llm_invoker,
        )

        result = await worker.run("TEST", "2026-01-01", "2026-02-01")
        thesis = result["result"]["thesis"]

        assert result["result"]["score"] == 0.0
        assert result["result"]["evidence_nodes"] == []
        assert thesis.count(SYSTEM_NOTE) == 1

    asyncio.run(_run())


def test_spec_3_1_neutralizes_non_neutral_score_when_llm_returns_empty_evidence_list():
    async def _run():
        mod = import_worker_nodes(force_reload=True)

        def fake_rubric_fetcher(_rubric_id):
            return {"criteria": "empty-evidence-list"}

        async def fake_document_fetcher(_ticker, _doc_type, _date_range):
            return {"status": "available", "data": "Verified excerpt text only."}

        def fake_llm_invoker(_prompt, *, system):
            _ = system
            return json.dumps(
                {
                    "score": 0.9,
                    "thesis": "Strongly positive.",
                    "evidence_nodes": [],
                }
            )

        worker = mod.PENRSWorker(
            name="Empty Evidence Neutralizer",
            weight=1.0,
            signal_density=1.0,
            rubric_id="test_rubric",
            document_type=DocumentType.NEWS_SENTIMENT,
            rubric_fetcher=fake_rubric_fetcher,
            document_fetcher=fake_document_fetcher,
            llm_invoker=fake_llm_invoker,
        )

        result = await worker.run("TEST", "2026-01-01", "2026-02-01")

        assert result["result"]["evidence_nodes"] == []
        assert result["result"]["score"] == 0.0
        assert SYSTEM_NOTE in result["result"]["thesis"]

    asyncio.run(_run())


def test_spec_3_1_quote_validation_uses_truncated_excerpt_not_full_document():
    async def _run():
        mod = import_worker_nodes(force_reload=True)

        def fake_rubric_fetcher(_rubric_id):
            return {"criteria": "truncate-context"}

        # Quote appears only after the first 40 chars.
        long_text = "A" * 40 + " QUOTE_ONLY_IN_TAIL " + "B" * 40

        async def fake_document_fetcher(_ticker, _doc_type, _date_range):
            return {"status": "available", "data": long_text}

        def fake_llm_invoker(_prompt, *, system):
            _ = system
            return json.dumps(
                {
                    "score": 0.25,
                    "thesis": "Tail quote support.",
                    "evidence_nodes": [
                        {"verbatim_quote": "QUOTE_ONLY_IN_TAIL", "reasoning": "Present in full document only."}
                    ],
                }
            )

        worker = mod.PENRSWorker(
            name="Truncation Guard",
            weight=1.0,
            signal_density=1.0,
            rubric_id="test_rubric",
            document_type=DocumentType.NEWS_SENTIMENT,
            rubric_fetcher=fake_rubric_fetcher,
            document_fetcher=fake_document_fetcher,
            llm_invoker=fake_llm_invoker,
            max_context_chars=40,
        )

        result = await worker.run("TEST", "2026-01-01", "2026-02-01")

        assert result["result"]["evidence_nodes"] == []
        assert result["result"]["score"] == 0.0
        assert SYSTEM_NOTE in result["result"]["thesis"]

    asyncio.run(_run())

## Spec 3 Arbiter, Report and Master Integration Tests

In [ ]:
import re
import tempfile


def test_spec_3_2_arbiter_allows_neutral_score_without_evidence_nodes():
    arbiter = import_orchestrator(force_reload=True).ArbiterAgent()
    worker_result = {
        "status": "available",
        "worker": {"name": "w1", "weight": 1.0, "signal_density": 0.7},
        "result": {"score": 0.0, "thesis": "neutral"},
    }
    arbiter._validate_worker_result(worker_result)


def test_spec_3_2_arbiter_rejects_non_neutral_score_without_evidence_nodes():
    arbiter = import_orchestrator(force_reload=True).ArbiterAgent()
    worker_result = {
        "status": "available",
        "worker": {"name": "w1", "weight": 1.0, "signal_density": 0.7},
        "result": {"score": 0.2, "thesis": "neutral"},
    }
    try:
        arbiter._validate_worker_result(worker_result)
        assert False, "Expected ValueError"
    except ValueError as exc:
        assert re.search("Non-neutral worker score requires validated evidence_nodes", str(exc))


def test_spec_3_2_arbiter_rejects_non_neutral_score_with_empty_evidence_nodes():
    arbiter = import_orchestrator(force_reload=True).ArbiterAgent()
    worker_result = {
        "status": "available",
        "worker": {"name": "w1", "weight": 1.0, "signal_density": 0.7},
        "result": {"score": -0.3, "thesis": "neutral", "evidence_nodes": []},
    }
    try:
        arbiter._validate_worker_result(worker_result)
        assert False, "Expected ValueError"
    except ValueError as exc:
        assert re.search("Non-neutral worker score requires validated evidence_nodes", str(exc))


def test_spec_3_2_arbiter_accepts_non_neutral_score_with_validated_evidence_nodes():
    arbiter = import_orchestrator(force_reload=True).ArbiterAgent()
    worker_result = {
        "status": "available",
        "worker": {"name": "w1", "weight": 1.0, "signal_density": 0.7},
        "result": {
            "score": 0.5,
            "thesis": "neutral",
            "evidence_nodes": [{"verbatim_quote": "in excerpt", "reasoning": "validated"}],
        },
    }
    arbiter._validate_worker_result(worker_result)


def test_spec_3_2_arbiter_rejects_non_neutral_string_score_without_evidence_nodes():
    arbiter = import_orchestrator(force_reload=True).ArbiterAgent()
    worker_result = {
        "status": "available",
        "worker": {"name": "w1", "weight": 1.0, "signal_density": 0.7},
        "result": {"score": "0.2", "thesis": "neutral"},
    }
    try:
        arbiter._validate_worker_result(worker_result)
        assert False, "Expected ValueError"
    except ValueError as exc:
        assert re.search("Non-neutral worker score requires validated evidence_nodes", str(exc))


def test_spec_3_2_penrs_report_model_exposes_evidence_field_with_default_factory():
    mod = import_orchestrator(force_reload=True)
    report = mod.PENRSReport.model_validate(
        {
            "ticker": "MRNA",
            "date_from": "2026-01-01",
            "date_to": "2026-02-01",
            "generated_at": "2026-03-06T00:00:00+00:00",
            "worker_results": [],
            "arbiter": {"status": "available"},
            "master": {"status": "available"},
            "report_path": "/tmp/r.json",
        }
    )
    assert report.evidence == []


def test_spec_3_2_penrs_report_evidence_default_factory_not_shared_between_instances():
    mod = import_orchestrator(force_reload=True)
    payload = {
        "ticker": "MRNA",
        "date_from": "2026-01-01",
        "date_to": "2026-02-01",
        "generated_at": "2026-03-06T00:00:00+00:00",
        "worker_results": [],
        "arbiter": {"status": "available"},
        "master": {"status": "available"},
        "report_path": "/tmp/r.json",
    }
    report_a = mod.PENRSReport.model_validate(payload)
    report_b = mod.PENRSReport.model_validate(payload)

    report_a.evidence.append({"verbatim_quote": "q"})
    assert report_b.evidence == []


def test_spec_3_3_master_synthesize_maps_evidence_with_derived_cache_keys():
    mod = import_orchestrator(force_reload=True)
    master = mod.MasterAgent()
    worker_results = [
        {
            "status": "available",
            "worker": {"name": "Clinical Signals", "weight": 1.0, "signal_density": 0.8},
            "document_type": "earnings_call",
            "result": {
                "score": 0.3,
                "evidence_nodes": [
                    {"verbatim_quote": "trial enrollment improved", "reasoning": "quoted"},
                    "invalid-node",
                ],
            },
        },
        {
            "status": "available",
            "worker": {"name": "   ", "weight": 1.0, "signal_density": 0.8},
            "result": {
                "score": -0.1,
                "evidence_nodes": [{"verbatim_quote": "cash burn narrowed", "reasoning": "quoted"}],
            },
        },
        {
            "status": "error",
            "worker": {"name": "Should Skip", "weight": 1.0, "signal_density": 0.8},
            "document_type": "press_release",
            "result": {
                "score": 0.8,
                "evidence_nodes": [{"verbatim_quote": "must be ignored", "reasoning": "not available"}],
            },
        },
    ]
    synthesized = master.synthesize(
        ticker="MRNA",
        date_from="2026-01-01",
        date_to="2026-02-01",
        worker_results=worker_results,
        arbiter_result={"weighted_score": 0.42},
    )
    evidence = synthesized["evidence"]
    assert len(evidence) == 2
    assert evidence[0]["cache_key"] == utils.cache_key(api="clinical_signals", ticker="MRNA", doc_type="earnings_call")
    assert evidence[1]["cache_key"] == utils.cache_key(api="unknown_worker", ticker="MRNA", doc_type="unknown_doc_type")
    assert synthesized["available_worker_count"] == 2
    assert synthesized["total_worker_count"] == 3


def test_spec_3_3_run_penrs_hoists_master_evidence_to_top_level_and_saved_report():
    async def _run():
        mod = import_orchestrator(force_reload=True)

        class StubWorker:
            name = "Evidence Worker"
            weight = 1.0
            signal_density = 0.9

            async def run(self, ticker, date_from, date_to):
                return {
                    "status": "available",
                    "ticker": ticker,
                    "date_from": date_from,
                    "date_to": date_to,
                    "worker": {"name": self.name, "weight": self.weight, "signal_density": self.signal_density},
                    "document_type": "news_sentiment",
                    "result": {
                        "score": 0.4,
                        "thesis": "Supported by evidence",
                        "evidence_nodes": [
                            {"verbatim_quote": "management reaffirmed guidance", "reasoning": "explicit statement"}
                        ],
                    },
                }

        with tempfile.TemporaryDirectory() as tmp_dir:
            report = await mod.run_penrs(
                "BIIB",
                "2026-01-01",
                "2026-02-01",
                workers=[StubWorker()],
                report_dir=tmp_dir,
            )

            assert "evidence" in report
            assert report["evidence"] == report["master"]["evidence"]
            assert len(report["evidence"]) == 1
            report_path = Path(report["report_path"])
            assert report_path.exists()
            payload = json.loads(report_path.read_text(encoding="utf-8"))
            assert payload["evidence"] == payload["master"]["evidence"]
            assert payload["evidence"] == report["evidence"]

    asyncio.run(_run())


def test_spec_3_3_run_penrs_sets_arbiter_error_when_worker_has_non_neutral_score_without_evidence():
    async def _run():
        mod = import_orchestrator(force_reload=True)

        class InvalidWorker:
            name = "Invalid Worker"
            weight = 1.0
            signal_density = 0.7

            async def run(self, ticker, date_from, date_to):
                return {
                    "status": "available",
                    "ticker": ticker,
                    "date_from": date_from,
                    "date_to": date_to,
                    "worker": {"name": self.name, "weight": self.weight, "signal_density": self.signal_density},
                    "document_type": "news_sentiment",
                    "result": {"score": 0.55, "thesis": "No evidence attached."},
                }

        with tempfile.TemporaryDirectory() as tmp_dir:
            report = await mod.run_penrs(
                "BIIB",
                "2026-01-01",
                "2026-02-01",
                workers=[InvalidWorker()],
                report_dir=tmp_dir,
            )

            assert report["arbiter"]["status"] == "error"
            assert "Non-neutral worker score requires validated evidence_nodes" in report["arbiter"]["error"]
            assert report["arbiter"]["weighted_score"] == 0.0
            assert report["master"]["final_score"] == 0.0
            assert report["master"]["evidence"] == []
            assert report["evidence"] == []

    asyncio.run(_run())


def test_spec_3_3_run_penrs_with_real_worker_neutralizes_hallucinated_quotes_before_arbiter():
    async def _run():
        orchestrator_mod = import_orchestrator(force_reload=True)
        worker_mod = import_worker_nodes(force_reload=True)

        def fake_rubric_fetcher(_rubric_id):
            return {"criteria": "integration-hallucination-destruction"}

        async def fake_document_fetcher(_ticker, _doc_type, _date_range):
            return {"status": "available", "data": "Validated excerpt has no fabricated quote."}

        def fake_llm_invoker(_prompt, *, system):
            _ = system
            return json.dumps(
                {
                    "score": -0.6,
                    "thesis": "Strong downside with bad citation.",
                    "evidence_nodes": [
                        {"verbatim_quote": "fabricated quote text", "reasoning": "hallucinated"},
                    ],
                }
            )

        worker = worker_mod.PENRSWorker(
            name="Quote Validator",
            weight=1.0,
            signal_density=0.9,
            rubric_id="r1",
            document_type=DocumentType.NEWS_SENTIMENT,
            rubric_fetcher=fake_rubric_fetcher,
            document_fetcher=fake_document_fetcher,
            llm_invoker=fake_llm_invoker,
        )

        with tempfile.TemporaryDirectory() as tmp_dir:
            report = await orchestrator_mod.run_penrs(
                "MRNA",
                "2026-01-01",
                "2026-02-01",
                workers=[worker],
                report_dir=tmp_dir,
            )

            worker_result = report["worker_results"][0]["result"]
            assert worker_result["score"] == 0.0
            assert worker_result["evidence_nodes"] == []
            assert "Score neutralized due to hallucinated evidence." in worker_result["thesis"]
            assert report["arbiter"]["status"] == "available"
            assert report["arbiter"]["weighted_score"] == 0.0
            assert report["master"]["evidence"] == []

    asyncio.run(_run())

## Test Runner

In [ ]:
# test_*.py source files have been removed; test functions are now defined inline above.
# Tests that require pytest fixtures (monkeypatch) need ipytest or a live pytest session.
# The tests below do not use fixtures and can be called directly:

import traceback

_direct_tests = [
    test_system_prompt_contains_required_role_and_mandatory_contradictions,
    test_arbiter_validates_required_worker_schema_fields,
    test_arbiter_clamps_scores_and_applies_star_rating_weights,
    test_arbiter_returns_json_schema_with_contradiction_flags_and_severities,
    test_document_type_is_strict_enum_with_nine_values,
    test_penrs_fetch_document_requires_document_type_enum,
    test_penrs_fetch_document_aggregates_multi_source_results,
    test_penrs_fetch_document_returns_not_released_with_attempted_apis,
    test_penrs_fetch_document_handles_partial_failures,
    test_truncate_for_context_preserves_start_and_end,
    test_parse_json_response_handles_markdown_block,
    test_parse_json_response_handles_embedded_prose_and_invalid_json_fallback,
    test_run_fetches_rubric_and_document_and_enriches_metadata,
    test_run_handles_not_released_without_llm_call,
    test_run_penrs_executes_pipeline_and_saves_report,
    test_run_penrs_worker_failure_is_isolated,
    # Worker Spec 2 unit tests
    test_spec_2_1_build_prompt_injects_strict_schema_and_verbatim_constraint,
    test_spec_2_2_parse_json_response_defaults_when_keys_missing,
    test_spec_2_2_parse_json_response_defaults_when_any_required_key_missing,
    test_spec_2_2_parse_json_response_enforces_types_and_clamps_score,
    test_spec_2_2_parse_json_response_coerces_score_to_float_type,
    test_spec_2_2_parse_json_response_supports_embedded_and_fenced_json,
    test_spec_2_2_parse_json_response_returns_default_for_non_json,
    test_spec_2_2_parse_json_response_handles_dict_and_none_inputs,
    # Worker Spec 2 integration tests
    test_spec_2_3_run_passes_anthropic_system_prompt_and_parses_response,
    test_spec_2_3_run_falls_back_when_invoker_rejects_system_kwarg,
    test_spec_2_3_default_llm_invoker_accepts_system_and_returns_schema_safe_payload,
    test_spec_2_3_run_passes_system_prompt_to_async_kwargs_invoker,
    # Fetcher unit tests (spec 1.1–1.5)
    test_spec_1_1_required_imports_and_mcp_tool_registration_present,
    test_spec_1_2_alpha_vantage_success_routes_and_caches,
    test_spec_1_2_alpha_vantage_error_skips_cache,
    test_spec_1_3_sec_edgar_success_routes_headers_and_caches,
    test_spec_1_3_sec_edgar_error_skips_cache,
    test_spec_1_4_openfda_without_api_key_routes_and_caches,
    test_spec_1_4_openfda_includes_api_key_when_present,
    test_spec_1_4_openfda_error_skips_cache,
    test_spec_1_5_pubmed_without_api_key_routes_and_caches,
    test_spec_1_5_pubmed_includes_api_key_when_present,
    test_spec_1_5_pubmed_error_skips_cache,
    # Fetcher cache integration tests (spec 1.2–1.5)
    test_spec_1_2_alpha_vantage_success_writes_immutable_ground_truth_to_disk,
    test_spec_1_3_sec_edgar_success_writes_expected_cache_record,
    test_spec_1_4_openfda_success_writes_expected_cache_record,
    test_spec_1_5_pubmed_success_writes_normalized_cache_record,
    test_error_responses_do_not_write_cache_files,
    # Spec 3 worker hallucination unit tests
    test_spec_3_1_prunes_hallucinated_nodes_and_keeps_valid_quotes,
    test_spec_3_1_neutralizes_score_and_appends_system_note_when_all_nodes_are_hallucinated,
    test_spec_3_1_system_note_is_not_duplicated_when_already_present,
    test_spec_3_1_neutralizes_non_neutral_score_when_llm_returns_empty_evidence_list,
    test_spec_3_1_quote_validation_uses_truncated_excerpt_not_full_document,
    # Spec 3 arbiter and report unit tests
    test_spec_3_2_arbiter_allows_neutral_score_without_evidence_nodes,
    test_spec_3_2_arbiter_rejects_non_neutral_score_without_evidence_nodes,
    test_spec_3_2_arbiter_rejects_non_neutral_score_with_empty_evidence_nodes,
    test_spec_3_2_arbiter_accepts_non_neutral_score_with_validated_evidence_nodes,
    test_spec_3_2_arbiter_rejects_non_neutral_string_score_without_evidence_nodes,
    test_spec_3_2_penrs_report_model_exposes_evidence_field_with_default_factory,
    test_spec_3_2_penrs_report_evidence_default_factory_not_shared_between_instances,
    # Spec 3 master and run_penrs integration tests
    test_spec_3_3_master_synthesize_maps_evidence_with_derived_cache_keys,
    test_spec_3_3_run_penrs_hoists_master_evidence_to_top_level_and_saved_report,
    test_spec_3_3_run_penrs_sets_arbiter_error_when_worker_has_non_neutral_score_without_evidence,
    test_spec_3_3_run_penrs_with_real_worker_neutralizes_hallucinated_quotes_before_arbiter,
]

_passed = _failed = 0
for _fn in _direct_tests:
    try:
        _fn()
        print(f"PASS  {_fn.__name__}")
        _passed += 1
    except Exception as _exc:
        print(f"FAIL  {_fn.__name__}: {_exc}")
        traceback.print_exc()
        _failed += 1

print(f"\n{_passed} passed, {_failed} failed")
print("(Tests using monkeypatch require ipytest: pip install ipytest)")